# Занятие 10 — Реляционная модель и SQL

## Подготовка

In [ ]:
from pathlib import Path
import json
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
DATA = Path("../data")
WORK = Path("../results")
WORK.mkdir(exist_ok=True)
decisions = pd.read_csv(DATA / "decisions.csv", dtype={"lei": "string", "county_code": "string"})
assert len(decisions) == 6830
assert decisions["source_row"].is_unique
rng = np.random.default_rng(2026)
import sqlite3
connection = sqlite3.connect(":memory:")
decisions.to_sql("decisions", connection, index=False, if_exists="replace")
organizations = pd.read_csv(DATA / "organizations.csv", dtype={"lei": "string"})
organizations[["lei", "activity_year", "organization_name"]].to_sql("organizations", connection, index=False, if_exists="replace")
def query(sql, params=()):
    return pd.read_sql_query(sql, connection, params=params)


## B01 — слайд 1: Те же данные, другой способ запроса

In [ ]:
print(query("SELECT source_row, lei, activity_year, action_taken FROM decisions LIMIT 5"))
print(query("SELECT COUNT(*) AS n FROM decisions"))

## B02 — слайд 2: Ключи связывают таблицы

In [ ]:
duplicate_keys = query("""SELECT lei, activity_year, COUNT(*) AS n
FROM organizations GROUP BY lei, activity_year HAVING COUNT(*) > 1""")
assert duplicate_keys.empty
assert query("SELECT COUNT(*) AS n FROM decisions WHERE source_row IS NULL").loc[0, "n"] == 0

## B03 — слайд 3: SELECT и AS

In [ ]:
print(query("""SELECT source_row, loan_amount,
loan_amount / 1000.0 AS loan_kusd
FROM decisions ORDER BY source_row LIMIT 5"""))

## B04 — слайд 4: WHERE: условие для строки

In [ ]:
selected_sql = query("""SELECT source_row FROM decisions
WHERE action_taken = 3 AND loan_amount > 1000000 ORDER BY source_row""")
selected_pd = decisions.loc[decisions["action_taken"].eq(3) & decisions["loan_amount"].gt(1000000), "source_row"].sort_values()
assert selected_sql["source_row"].tolist() == selected_pd.tolist()
print(len(selected_sql))

## B05 — слайд 5: NULL: неизвестное значение

In [ ]:
null_check = query("""SELECT COUNT(*) AS rows_n, COUNT(income) AS known_n,
COUNT(*) - COUNT(income) AS missing_n FROM decisions""")
assert null_check.loc[0, "missing_n"] == 342
print(null_check)

## B06 — слайд 6: ORDER BY и LIMIT

In [ ]:
top = query("""SELECT source_row, loan_amount FROM decisions
ORDER BY loan_amount DESC, source_row ASC LIMIT 5""")
print(top)
check_top = decisions.sort_values(["loan_amount", "source_row"], ascending=[False, True]).head(5)
assert top["source_row"].tolist() == check_top["source_row"].tolist()


## B07 — слайд 7: GROUP BY: одна строка на группу

In [ ]:
outcomes_sql = query("""SELECT action_taken, COUNT(*) AS n,
AVG(loan_amount) AS mean_amount FROM decisions
GROUP BY action_taken ORDER BY action_taken""")
assert outcomes_sql["n"].sum() == len(decisions)
print(outcomes_sql)

## B08 — слайд 8: HAVING: условие для готовой группы

In [ ]:
large_orgs = query("""SELECT lei, activity_year, COUNT(*) AS n
FROM decisions GROUP BY lei, activity_year
HAVING COUNT(*) >= 20 ORDER BY n DESC, lei""")
assert large_orgs["n"].ge(20).all()
print(large_orgs.head())

## B09 — слайд 9: CASE: условное вычисление

In [ ]:
profile_sql = query("""SELECT lei, activity_year, COUNT(*) AS n_decisions,
SUM(CASE WHEN action_taken = 3 THEN 1 ELSE 0 END) AS n_denied,
AVG(CASE WHEN action_taken = 3 THEN 1.0 ELSE 0.0 END) AS denial_rate
FROM decisions GROUP BY lei, activity_year""")
assert np.allclose(profile_sql["denial_rate"], profile_sql["n_denied"] / profile_sql["n_decisions"])
assert profile_sql["n_denied"].sum() == 691
print(profile_sql.head())

## B10 — слайд 10: JOIN: поиск совпадения по ключу

In [ ]:
enriched_sql = query("""SELECT d.source_row, d.lei, d.activity_year,
d.action_taken, o.organization_name FROM decisions AS d
LEFT JOIN organizations AS o
ON d.lei = o.lei AND d.activity_year = o.activity_year
ORDER BY d.source_row""")
assert len(enriched_sql) == len(decisions)
assert enriched_sql["source_row"].is_unique
print(enriched_sql.head())

## B12 — слайд 12: Параметры вместо подстановки текста

In [ ]:
parameters = (3, 1000000)
parameterized = query("""SELECT source_row FROM decisions
WHERE action_taken = ? AND loan_amount > ? ORDER BY source_row""", parameters)
assert parameterized.equals(selected_sql)

## B13 — слайд 13: Сверяем SQL и pandas

In [ ]:
profile_pd = decisions.assign(is_denied=decisions["action_taken"].eq(3)).groupby(["lei", "activity_year"]).agg(
    n_decisions=("source_row", "size"), n_denied=("is_denied", "sum")).reset_index()
a = profile_sql.sort_values(["lei", "activity_year"]).reset_index(drop=True)
b = profile_pd.sort_values(["lei", "activity_year"]).reset_index(drop=True)
assert a["lei"].tolist() == b["lei"].tolist()
assert np.array_equal(a[["n_decisions", "n_denied"]], b[["n_decisions", "n_denied"]])
profile_sql.to_csv(WORK / "sql_profile.csv", index=False)